In [0]:
# ========================================
# Silver Layer: Regions with Data Quality
# ========================================

from pyspark.sql.functions import col, trim, upper, current_timestamp, when, length

# Read from Bronze
df_bronze = spark.read.table("ecommerce.e_comm_bronze.tblorders")

print(f"Bronze record count: {df_bronze.count()}")

# ── Data Quality Step 1: Standardize and clean text ──
df_cleaned = df_bronze \
    .withColumn("order_id", trim(col("product_id"))) \
    .withColumn("customer_id", trim(upper(col("customer_id")))) \
    .withColumn("product_id", trim(upper(col("product_id")))) \
    

# ── Data Quality Step 2: Remove duplicates ──
df_deduped = df_cleaned.dropDuplicates(["order_id"])
df_deduped.createOrReplaceTempView("vw_df_deduped")
print(f"After deduplication: {df_deduped.count()}")

# ── Data Quality Step 3: Add surrogate_key and data quality flag ──
df_with_dq = spark.sql("""
        select 
        row_number() over (order by order_id) as order_key,
        order_id,
        customer_id,
        product_id, 
        order_date,
        quantity,
        total_amount,
        case when quantity < 0 then "quantity_is_negative"        
            when total_amount < 0 then "total_amount_is_negative"
            else "is_valid" end as dq_note ,
        current_timestamp() as load_ts
        from vw_df_deduped 
                       """)

# Show data quality summary
print("\n=== Data Quality Summary ===")
print(f"Total records: {df_with_dq.count()}")
print(f"Valid records: {df_with_dq.filter(col('dq_note') == "is_valid").count()}")
print(f"Invalid records: {df_with_dq.filter(col('dq_note') != "is_valid").count()}")

# Show sample of invalid records if any exist
invalid_records = df_with_dq.filter(col('dq_note') != "is_valid")
if invalid_records.count() > 0:
    print("\nSample invalid records:")
    invalid_records.show(25, truncate=False)

# Create temp view for querying
df_with_dq.createOrReplaceTempView("vw_products")

print("\n✅ Silver transformation complete with data quality checks")

In [0]:
#write data to a delta table
df_with_dq \
    .write \
        .format("delta") \
            .option("overwriteSchema", "true") \
                .mode("overwrite").saveAsTable("ecommerce.e_comm_silver.orders")